#**Actividad: Predicción del Movimiento de un Paracaidista con Redes Neuronales Recurrentes (RNN)**

## Objetivo

Aplicar redes neuronales recurrentes (RNN), específicamente con RNR Simples y con LSTM, para modelar y predecir el comportamiento dinámico de un paracaidista en caída libre con resistencia del aire, usando una serie de tiempo simulada.

---

## Contexto físico

El movimiento vertical de un paracaidista está gobernado por una ecuación diferencial ordinaria (EDO). Esta solución representa cómo la velocidad del paracaidista evoluciona hasta alcanzar una velocidad terminal `v_terminal = mg/k`.

---

## Descripción general del taller

En este taller vas a:

1. Simular los datos de velocidad de un paracaidista a partir del modelo físico.
2. Construir un conjunto de datos en forma de serie de tiempo.
3. Usar una red neuronal LSTM para predecir la siguiente velocidad en la serie.
4. Evaluar el modelo y analizar los resultados desde el punto de vista físico y computacional.

---

##Instrucciones

### 1. Simulación del sistema físico
- Elige valores realistas para los parámetros del modelo:  
  - Masa `m = 80 kg`  
  - Coeficiente de fricción `k = 12 kg/s`  
- Simula la velocidad desde `t = 0` hasta `t = 20 s`, con pasos de `0.1 s`.

### 2. Preparación de la serie de tiempo
- Normaliza la serie de datos.
- Divide la serie en ventanas deslizantes de longitud fija (por ejemplo, 10 pasos).
- Cada entrada del modelo será una secuencia de 10 velocidades, y la salida será la velocidad siguiente.

### 3. Construcción y entrenamiento del modelo
- Usa Keras para construir una red LSTM con la siguiente arquitectura recomendada:

```python
model = Sequential()
model.add(LSTM(64, input_shape=(window_size, 1)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
```
---
### 4. Entrena el modelo y registra el historial de pérdida.
Evaluación del modelo
- Compare gráficamente las velocidades predichas frente a las reales.
- Calcule métricas de error como el MSE (Error cuadrático medio).
- Discute las diferencias y posibles fuentes de error.
- Simule otros escenarios con diferentes masas y coeficientes de fricción.
- Ajusta los hiperparámetros del modelo (número de unidades LSTM, tamaño de ventana, etc.).
- Pruebe con otros tipos de RNN (por ejemplo, GRU).

---
### 5.Preguntas
- ¿Hasta qué punto la LSTM logra capturar el comportamiento físico del sistema?
- ¿Qué representa la velocidad terminal y cómo la identifica el modelo?
- ¿Qué ventajas ofrece una red LSTM frente a una red densa tradicional?
- ¿Cómo cambiarían los resultados si los datos fueran medidos experimentalmente con ruido?

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from ipywidgets import interact, IntSlider, FloatSlider

# --- Función interactiva principal ---
def entrenar_y_graficar(window_size=10, epochs=50, neurons=50, m=80.0, k=12.0):
    g = 9.8  # gravedad m/s^2
    ti, tf_ = 0.0, 20.0
    h = 0.1
    n = int((tf_ - ti) / h) + 1

    # --- Cálculos físicos ---
    vt = np.sqrt((m * g) / k)
    tiempo = np.linspace(ti, tf_, n)
    exp_arg = (g / vt) * tiempo
    velocidad = vt * ((np.exp(exp_arg) - np.exp(-exp_arg)) / (np.exp(exp_arg) + np.exp(-exp_arg)))
    entrada = pd.DataFrame({'Tiempo (s)': tiempo, 'Velocidad (m/s)': velocidad})

    # --- Normalización y creación de ventanas ---
    velocidades = entrada['Velocidad (m/s)'].values.reshape(-1, 1)
    scaler = MinMaxScaler()
    velocidades_scaled = scaler.fit_transform(velocidades)

    X, y = [], []
    for i in range(len(velocidades_scaled) - window_size):
        X.append(velocidades_scaled[i:i+window_size])
        y.append(velocidades_scaled[i+window_size])
    X, y = np.array(X), np.array(y)

    # División entrenamiento / prueba
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    # --- Modelo LSTM ---
    model = Sequential([
        LSTM(neurons, activation='tanh', input_shape=(window_size, 1)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')

    # --- Entrenamiento ---
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=32, verbose=0, validation_split=0.1)

    # --- Evaluación ---
    y_pred_scaled = model.predict(X_test)
    y_pred = scaler.inverse_transform(y_pred_scaled)
    y_test_inv = scaler.inverse_transform(y_test)

    # Cálculo de errores
    mse = mean_squared_error(y_test_inv, y_pred)
    mae = mean_absolute_error(y_test_inv, y_pred)

    # --- Información en consola ---
    print(f"\n=== Parámetros del experimento ===")
    print(f"Masa = {m:.1f} kg")
    print(f"Coeficiente de fricción = {k:.1f} kg/s")
    print(f"Velocidad terminal (vt) = {vt:.2f} m/s")
    print(f"Ventana = {window_size} pasos, Épocas = {epochs}, Neuronas LSTM = {neurons}")
    print(f"\n--- Métricas de error ---")
    print(f"Error cuadrático medio (MSE): {mse:.6f}")
    print(f"Error absoluto medio (MAE): {mae:.6f}")

    # --- Predicciones sobre toda la serie ---
    pred_scaled_full = model.predict(X)
    pred_full = scaler.inverse_transform(pred_scaled_full)
    tiempo_pred = tiempo[window_size:]

    # --- Gráfico principal ---
    plt.figure(figsize=(12, 6))
    plt.plot(tiempo, velocidad, label='Velocidad Real (Analítica)', color='purple')
    plt.plot(tiempo_pred, pred_full.flatten(), label='Predicción LSTM', color='orange', linestyle='--')
    plt.axhline(y=vt, color='green', linestyle=':', label=f'vₜ = {vt:.2f} m/s')
    plt.title('Predicción de la Velocidad del Paracaidista con LSTM vs. Solución Analítica')
    plt.xlabel('Tiempo (s)')
    plt.ylabel('Velocidad (m/s)')
    plt.legend()
    plt.grid(True)
    plt.show()

    # --- Historial de pérdida ---
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Entrenamiento')
    plt.plot(history.history['val_loss'], label='Validación')
    plt.title('Evolución de la pérdida durante el entrenamiento')
    plt.xlabel('Época')
    plt.ylabel('Pérdida (MSE)')
    plt.legend()
    plt.grid(True)
    plt.show()

# --- Widgets interactivos ---
interact(
    entrenar_y_graficar,
    window_size=IntSlider(value=10, min=5, max=30, step=1, description='Ventana'),
    epochs=IntSlider(value=50, min=10, max=200, step=10, description='Épocas'),
    neurons=IntSlider(value=50, min=10, max=100, step=10, description='Neuronas'),
    m=FloatSlider(value=80.0, min=40.0, max=120.0, step=5.0, description='Masa (kg)'),
    k=FloatSlider(value=12.0, min=5.0, max=20.0, step=1.0, description='Fricción (kg/s)')
);


interactive(children=(IntSlider(value=10, description='Ventana', max=30, min=5), IntSlider(value=50, descripti…

### 5. Preguntas

- ¿Hasta qué punto la LSTM logra capturar el comportamiento físico del sistema?

La LSTM logra ajustar muy bien el comportamiento del sistema a medida que se ajustan parámetros de manera interactiva gracias a ipywidgets; según el parámetro de pérdida podemos evidenciar un buen ajuste gracias al número de neuronas y las época que le toma a entrenar a la red.
Algo a destacar es el ajuste que se le hace al batch como parámetro de entrenamiento ya que analiza la cantidad de dat disponible. El código permite una visualización amplia para ajustes de parámetros de manera interactiva y ver un comportamiento en casi tiempo real de como aprende y ajusta la red. Se deja un batch size por defecto en 32 para mejor toma de datos de entrenamiento.

---
- ¿Qué representa la velocidad terminal y cómo la identifica el modelo?

La velocidad terminal implica un punto de cambio cero en la funcion, el modelo reconoce que llegado a un punto no cambia el resultado pero le cuesta ser preciso con el punto exacto donde ocurre, por lo que llega a una velocidad terminal mayor según los parámetros da masa de 90 kg, una venta de 1 pasos, 50 épocas y 50 neuronas.

Al aumentar el número de neuronas y las épocas vemos mejor ajuste de la predicción de la red neuronal con el comportamiento esperado en el modelo.

---
- ¿Qué ventajas ofrece una red LSTM frente a una red densa tradicional?

La red tradicional tendria inmensos problemas para aprender el comportamiento del sistema una vez ha llegado a su velocidad terminal, pues no tendria como darse cuenta de que lleva tiempo en dicho comportamiento, se necesitaria overfitting para que lo predijera. Razón por la que al utilizar LSTM podemos ajustar el número de ventanas para que la predicción sea más adecuada.

---
- ¿Cómo cambiarían los resultados si los datos fueran medidos experimentalmente con ruido?

En dado caso la red podria aprenderse los cambios aleatorios del comportamiento y mostrar que la velocidad terminal cambia ligeramente con saltos pequeños y se mantiene alrededor de un punto, esto es lo que podría desembocar el overfitting que precisamente evitamos que se genere, no aprenda datos si no que identifique patrones en nuevos datos para la red. Algunos ejemplo es el ruido gaussiano o salt-pepper para imágenes.

---

Queda pendiente la comparacion con otras redes neurnales como GRUB o una Simple RNN